In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import logging
import random
from copy import deepcopy
import re
from sklearn.metrics import mean_squared_error

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import numpy as np
from dotenv import load_dotenv
from natsort import natsorted

from halcyon.Simulators.SimulationResult import SimulationResult
from halcyon.results.manipulate_results_df import (create_tidy_df_from_mlflow_runs, 
                                                   split_df_into_single_and_double_ko)

from halcyon.results import DfColumnNames

seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)
os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR'] = 'False'

In [ ]:
runs_to_download = []

runs_to_download.extend(mlflow.search_runs(filter_string="params.model_class = 'HybridModel'",
      experiment_ids=['11'], output_format='list'))
runs_to_download.extend(mlflow.search_runs(
      filter_string=f"attributes.status = 'FINISHED' and params.model_class = 'MLP' and tags.mlflow.parentRunId ='f02bf732b2ba4817b254bd5174f1549b'",
            experiment_ids=['6'], output_format='list'))
print(len(runs_to_download))

In [ ]:
raw_predictions_df = create_tidy_df_from_mlflow_runs(runs_to_download)
raw_predictions_df.head()

In [ ]:
raw_predictions_df.shape

In [ ]:
model_name_records = []
for run_id in raw_predictions_df['run_id'].unique():
    run = mlflow.get_run(run_id)
    model_name_records.append({
        'run_id': run_id,
        'model_name': run.data.tags['mlflow.runName'],
        'modelled_genes': run.data.params.get('modelled_gene_labels', None),
        'parameters_set_to_zero': run.data.params.get('parameters_set_to_zero', None) })
model_name_df = pd.DataFrame(model_name_records)
raw_predictions_df = pd.merge(raw_predictions_df, model_name_df, on='run_id')
raw_predictions_df.head()

In [ ]:
raw_predictions_df.model_name.value_counts()

In [ ]:
records = []
for run_id in raw_predictions_df['parent_run_id'].unique():
    ml_flow_download_path = mlflow.artifacts.download_artifacts(
        run_id=run_id, artifact_path="dataset"
    )
    sim_output = SimulationResult.load_from_pkl(
        Path(ml_flow_download_path) / "simulation_result.pkl"
    )
    sim_output.noiseless_for_testing = True
    long_form_simulation = sim_output.to_long_form_df()
    long_form_simulation[DfColumnNames.RUN_ID_COL] = run_id
    records.append(long_form_simulation)
long_form_simulation = pd.concat(records, ignore_index=True)

In [ ]:
records = []
for run_id in raw_predictions_df['parent_run_id'].unique():
    run = mlflow.get_run(run_id)
    records.append({
        'parent_run_id': run_id,
        'nr_genes': run.data.params['nr_genes'],
    })
nr_genes_df = pd.DataFrame(records)
nr_genes_df.head()

In [ ]:
outcome = pd.merge(
    raw_predictions_df, 
    long_form_simulation, 
    left_on=['parent_run_id', 'gene', 'time', 'condition'], 
    right_on=['run_id', 'gene', 'time', 'condition'], 
    validate='m:1',
    suffixes=['_from_pred_df', '_from_simulation'])

outcome = pd.merge(outcome, nr_genes_df, on='parent_run_id')

outcome.head()

In [ ]:
outcome.model_name.value_counts()

In [ ]:
outcome = outcome[['gene', 'time', 'predicted_expression', 
       'condition', 'run_id_from_pred_df', 'model_name', 'modelled_genes',
       'ground_truth_from_simulation', 'nr_genes', 'parameters_set_to_zero']]

outcome = outcome.rename(columns={"run_id_from_pred_df": "run_id", "ground_truth_from_simulation": "noiseless_ground_truth"})

In [ ]:
outcome['parameters_set_to_zero'].value_counts()

In [ ]:
single_ko_df, double_ko_df = split_df_into_single_and_double_ko(outcome)

In [ ]:
def annotate_kos_in_df(paired):
    paired = deepcopy(paired)
    def make_ko_lookup(df):
        unique_combos = df[['condition', 'modelled_genes']].drop_duplicates()
    
        lookup = {}
        for _, row in unique_combos.iterrows():
            cond = row['condition']
            genes = row['modelled_genes']
            key = (cond, genes)

            kos = re.findall(r"Node_\d+", cond)
        
            if all(f"'{ko}'" in genes for ko in kos):
                ko_in_modelled_genes = 'fully'
            elif any(f"'{ko}'" in genes for ko in kos):
                ko_in_modelled_genes = 'partially'
            else:
                ko_in_modelled_genes = 'none'
        
            lookup[key] = ko_in_modelled_genes
    
        return lookup

    lookup = make_ko_lookup(paired)
    paired['ko_in_mm'] = paired.apply(
        lambda row: lookup[(
            row['condition'],
            row['modelled_genes'] if row['modelled_genes'] is not None else None
        )],
    axis=1
)
    paired['gene_in_mm'] = paired.apply(lambda row:
             bool(re.search(rf"'{row['gene']}'", row['modelled_genes'])), axis=1)
    return paired


In [ ]:
double_ko_df.model_name.value_counts()

In [ ]:
groupby_list = [
    'model_name',
    'modelled_genes',
    'nr_genes',
    DfColumnNames.GENE_COL,
    DfColumnNames.CONDITION_COL,
    'parameters_set_to_zero'
]
mse_df = (
    double_ko_df.groupby(groupby_list, dropna=False)
    .apply(
        lambda g: mean_squared_error(
            g['noiseless_ground_truth'], g['predicted_expression']
        ),
        include_groups=False,
    )
    .reset_index(name=DfColumnNames.MSE_COL)
)

mlp_mse = mse_df[mse_df['model_name'] == 'MLP'][['gene', 'condition', 'MSE', 'nr_genes']].rename(columns={'MSE': 'mlp_mse'})
hybrid_mse = mse_df[mse_df['model_name'] == 'HybridModel'][['gene', 'condition', 'MSE', 'modelled_genes', 'nr_genes']].rename(columns={'MSE': 'hybrid_mse'})

paired = hybrid_mse.merge(mlp_mse, on=['gene', 'condition', 'nr_genes'])
paired = annotate_kos_in_df(paired)
paired

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def cached_check(gene_name, ko_string):
    return sim_output.network.get_ko_to_target_distance(gene_name, ko_string)

paired["KO-to-target distance"] = paired.apply(
    lambda row: cached_check(row["gene"], row["condition"]),
    axis=1)
paired['KO-to-target distance'].describe()

In [ ]:
# 1. Get the sorted, unique integers from your dataframe
unique_ints = sorted(paired['KO-to-target distance'].unique())

# 2. Generate a continuous palette with exactly that many colors
# "viridis", "Blues", "magma", etc., will all work here
gradient_colors = sns.color_palette("magma", n_colors=len(unique_ints))

# 3. Zip them into a dictionary: {1: color1, 2: color2, ...}
custom_palette = dict(zip(unique_ints, gradient_colors))

In [ ]:
epistasis_df = pd.read_csv('epistasis_levels.csv')

In [ ]:
paired.head()

In [ ]:
epistasis_df.head()

In [ ]:
paired = pd.merge(paired, epistasis_df[['condition', 'gene', 'single-to-double-ko-divergence']])

In [ ]:
paired.head()

In [ ]:
paired.sort_values(by='hybrid_mse', ascending=False).head()

In [ ]:
sns.set_theme()
sns.set_context("paper", font_scale=1.2)
sns.set_style('ticks')
palette = sns.color_palette()
g = sns.relplot(data=paired, x='mlp_mse', y='hybrid_mse', row='gene_in_mm', col='ko_in_mm',  hue='single-to-double-ko-divergence', palette='viridis',
                   s=20, height=3, aspect=1, facet_kws={'margin_titles': True}) #, hue_norm=(0, .4))
# g.figure.set_size_inches(6, 3) 
g.set(xscale="log", yscale="log")
for ax in g.axes.flat:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    for collection in ax.collections:
        collection.set_rasterized(True)
# plt.tight_layout()
# for text in g._margin_titles_texts:
#     text.set_visible(False)

# g.savefig("mlp_vs_hybrid_with_mistakes_log_hue_divergence.pdf", dpi=600, bbox_inches='tight')

In [ ]:
from matplotlib.colors import LogNorm, Normalize
import textwrap 

sns.set_theme()
sns.set_context("paper", font_scale=1.2)
sns.set_style('ticks')

min_scale = round(paired['single-to-double-ko-divergence'].quantile(0.01), 1)
max_scale = round(paired['single-to-double-ko-divergence'].quantile(0.99), 1)

palette = sns.color_palette()
cmap = 'viridis'

norm = Normalize(min_scale, max_scale)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])


g = sns.relplot(data=paired, x='mlp_mse', y='hybrid_mse', row='gene_in_mm', col='ko_in_mm',
                hue='single-to-double-ko-divergence', palette=cmap, hue_norm=norm, legend=False,
                s=20, height=3, aspect=1, facet_kws={'margin_titles': True}) 
# g.figure.set_size_inches(6, 3) 
g.set(xscale="log", yscale="log")
for ax in g.axes.flat:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    for collection in ax.collections:
        collection.set_rasterized(True)

cbar = g.figure.colorbar(sm, 
                        ax=g.axes, 
                        shrink=0.5,       # scales the bar's length (0 to 1)
                        aspect=30,        # higher = thinner bar (default is ~20)
                        fraction=0.03,    # how much of the figure width it eats up
                        pad=0.15,         # gap between the bar and the last facet
                        )

wrapped_title = textwrap.fill('Single-double KO divergence', width=15)

cbar.ax.set_title(wrapped_title, fontsize=10, pad=8)
cbar.set_ticks(np.arange(min_scale, max_scale+0.05, 0.05))
# plt.tight_layout()
# for text in g._margin_titles_texts:
#     text.set_visible(False)

g.savefig("mlp_vs_hybrid_with_mistakes_log_hue_divergence.pdf", dpi=600, bbox_inches='tight')

In [ ]:
stdev_df = outcome.groupby(['gene', 'condition', 'nr_genes'])['noiseless_ground_truth'].std().reset_index().rename(
    columns={'noiseless_ground_truth': 'gene_expression_stdev'})
stdev_df.head()

In [ ]:
paired = pd.merge(paired, stdev_df, 'left', on=['gene', 'condition', 'nr_genes'])

In [ ]:
sns.set_theme()
sns.set_context("paper", font_scale=1.2)
sns.set_style('ticks')
palette = sns.color_palette()
g = sns.relplot(data=paired, x='mlp_mse', y='hybrid_mse', row='gene_in_mm', col='ko_in_mm', hue='KO-to-target distance', palette=custom_palette,
                #   legend=False, 
                  alpha=.75, s=20, height=3, aspect=1, facet_kws={'margin_titles': True})
# g.figure.set_size_inches(6, 3) 
# g.set(xscale="log", yscale="log")
for ax in g.axes.flat:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    for collection in ax.collections:
        collection.set_rasterized(True)
# plt.tight_layout()
# for text in g._margin_titles_texts:
#     text.set_visible(False)

g.savefig("mlp_vs_hybrid_with_mistakes_lin.pdf", dpi=600, bbox_inches='tight')

In [ ]:
sns.set_theme()
sns.set_context("paper", font_scale=1.2)
sns.set_style('ticks')
palette = sns.color_palette()
g = sns.relplot(data=paired, x='mlp_mse', y='hybrid_mse', row='gene_in_mm', col='ko_in_mm', hue='gene_expression_stdev', palette='magma',
                #   legend=False, 
                  alpha=.75, s=20, height=3, aspect=1, facet_kws={'margin_titles': True})
# g.figure.set_size_inches(6, 3) 
# g.set(xscale="log", yscale="log")
for ax in g.axes.flat:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    for collection in ax.collections:
        collection.set_rasterized(True)
# plt.tight_layout()
# for text in g._margin_titles_texts:
#     text.set_visible(False)

g.savefig("mlp_vs_hybrid_with_mistakes_lin_hue_stdev.pdf", dpi=600, bbox_inches='tight')